# 멀티모달 RAG Part 3: ColQwen 기반 시각적 검색

![colpali-architecture](https://cdn-uploads.huggingface.co/production/uploads/60f2e021adf471cbdf8bb660/La8vRJ_dtobqs6WQGKTzB.png)

- 이미지 출처: [HuggingFace Blog - ColPali](https://huggingface.co/blog/manu/colpali)

---

## 개요

**ColQwen**은 OCR 없이 문서 이미지를 직접 검색하는 **시각적 검색(Visual Retrieval)** 모델입니다.

기존 RAG 파이프라인:
```
PDF → OCR → 텍스트 추출 → 청킹 → 임베딩 → 벡터 DB → 검색
```

ColQwen 파이프라인:
```
PDF → 이미지 변환 → Vision Encoder → 멀티벡터 임베딩 → 검색
```

---

## 학습 목표

1. **Late Interaction (MaxSim)** 기반 멀티벡터 검색 원리 이해
2. **ColPali vs ColQwen** 아키텍처 차이점 파악
3. **byaldi/colpali-engine**을 활용한 문서 인덱싱 및 검색 구현
4. **End-to-End RAG 파이프라인** 구축 (검색 + VLM 답변 생성)

---

## 환경 설정

### Google Colab 실행 시

**T4 GPU (무료)** 환경에서 실행 가능합니다.

1. 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
2. 아래 셀 실행 시 자동으로 패키지 설치

### 로컬 환경 설정

```bash
# ColPali/ColQwen 엔진
uv add byaldi colpali-engine

# PDF 변환 도구
uv add pdf2image pillow

# Qwen2-VL 유틸리티
uv add qwen-vl-utils transformers accelerate

# GPU 최적화 (선택)
uv add bitsandbytes

# 시스템 요구사항 (poppler)
# macOS: brew install poppler
# Ubuntu: sudo apt-get install -y poppler-utils
```

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Google Colab 환경 감지됨")
    print("📦 의존성 해결 및 패키지 설치 중...")

    # 의존성 충돌을 피하기 위해 최우선적으로 torch 관련 버전 정렬
    !pip install -q --upgrade torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --extra-index-url https://download.pytorch.org/whl/cu121

    # 나머지 패키지 설치
    !pip install -q byaldi colpali-engine pdf2image pillow
    !pip install -q qwen-vl-utils transformers accelerate bitsandbytes

    # peft 버전 호환: transformers 4.57+ 호환성 고려
    !pip install -q --upgrade "peft>=0.17.0" "torchao>=0.16.0"

    # 시스템 유틸리티 설치
    !apt-get install -y poppler-utils > /dev/null 2>&1

    from google.colab import drive
    drive.mount('/content/drive')

    print("\n" + "="*40)
    print("✅ 모든 설정이 완료되었습니다!")
    print("⚠️  반드시 [런타임] -> [런타임 다시 시작]을 실행해주세요!")
    print("="*40)
else:
    print("💻 로컬 환경에서 실행 중")

### (1) 환경 변수 로드

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### (2) 기본 라이브러리

In [ ]:
import os
import io
from pathlib import Path
from pprint import pprint

import warnings
warnings.filterwarnings('ignore')

import sys
IN_COLAB = 'google.colab' in sys.modules

---

## 1. 이론적 배경

### 1.1 기존 RAG의 한계

> 🔰 **쉽게**: 기존 RAG는 "PDF를 글자로 변환(OCR) → 검색" 방식인데, 이 첫 단계에서 대부분 망가집니다.

**전통적 파이프라인**
```
PDF ─▶ OCR ─▶ 텍스트 ─▶ 임베딩 ─▶ 벡터DB ─▶ 검색
        ↑ 여기서 모든 문제 시작
```

**3대 문제점**

| 문제 | 현실 예시 |
|---|---|
| 🐢 **속도 병목** | 500p PDF OCR에 수십 분 |
| 😵 **한국어 품질** | 다양한 폰트·HWP→PDF 변환 시 인코딩 오류 |
| 📊 **시각 정보 손실** | 표 병합 구조, 차트 축, 수식 위·아래첨자 |

**ColQwen의 해법**: OCR 단계 제거 → **페이지 이미지 자체**를 임베딩. 사람이 문서를 눈으로 훑듯이 검색.

### 1.2 Retrieval 패러다임 스펙트럼

> 🔰 **쉽게**: 문서 검색은 "빠르지만 대충" ↔ "정확하지만 느림" 스펙트럼 위에 있고, **Late Interaction**은 그 **중간 균형점**입니다.

**3가지 방식 비교**

| 방식 | 대표 모델 | 정확도 | 지연 | 저장 |
|---|---|---|---|---|
| **Bi-Encoder** (단일 벡터) | BGE, text-embedding-3 | 중 | ⚡ 매우 낮음 | 낮음 |
| **Late Interaction** (토큰별) | ColBERT, **ColPali/ColQwen** | 높음 | 낮음 | **높음** |
| **Cross-Encoder** (통째) | BERT reranker, Cohere-rerank | 매우 높음 | 🐌 매우 높음 | — |

> 💡 **비유**
> - **Bi-Encoder**: 책 한 줄 요약만 보고 고르기
> - **Late Interaction**: 목차+소제목까지 보고 고르기
> - **Cross-Encoder**: 책 전체를 읽고 판단

**Late Interaction의 핵심 아이디어**
- 문서는 미리 **토큰별 벡터 행렬**로 인덱싱 (오프라인)
- 쿼리 시점엔 **MaxSim**(빠른 행렬곱)만 계산 → 토큰 정밀도 + 대규모 검색 양립

**계보**
```
ColBERT (SIGIR 2020)
  └─ ColBERTv2 (2022) — PLAID 압축
      └─ ColPali (arXiv:2407.01449, 2024) — OCR-free 문서검색
          └─ ColQwen (Qwen2-VL 기반)
              └─ ColQwen-Omni (2026) — 이미지+오디오+비디오
```

### 1.3 Late Interaction (MaxSim)

> 🔰 **쉽게**: 쿼리 토큰 하나하나가 문서에서 **각자 자기와 가장 비슷한 토큰**을 찾아 점수 매기고 합산합니다.

**수식**
$$\text{Score}(q, d) = \sum_{i \in q} \max_{j \in d} \text{sim}(q_i, d_j)$$

**그림으로 보면**
```
쿼리:  [어떤] [attention] [메커니즘]
          ↓       ↓            ↓    ← 각자 최고 점수 토큰 찾기
문서:  [이것은] [attention is] [어텐션은]
       0.3     0.9            0.7      → 합계: 1.9
```

> 💡 **비유**: 3명 친구가 뷔페에서 **각자 가장 좋아하는 토핑**을 고르고 점수 합산. 전체를 한 사람이 결정 X → 각자 최적 찾기.

**장점 (한 줄)**
- 토큰 단위 정밀 매칭 → 긴 문서·세부 구문 검색 강함
- 미분 가능·행렬곱 기반 → GPU에서 빠름

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

def maxsim_score(query_vecs, doc_vecs):
    """
    ColBERT-style MaxSim 연산

    Args:
        query_vecs: (n_query_tokens, dim) 쿼리 벡터
        doc_vecs: (n_doc_tokens, dim) 문서 벡터

    Returns:
        float: MaxSim 점수
    """
    similarity_matrix = cosine_similarity(query_vecs, doc_vecs)
    max_sims = similarity_matrix.max(axis=1)
    return max_sims.sum(), similarity_matrix


# 예제: "attention mechanism" 쿼리
query_vecs = np.array([
    [0.8, 0.2, 0.1],  # "attention"
    [0.3, 0.9, 0.2],  # "mechanism"
])

# 문서: "self-attention layer"
doc_vecs = np.array([
    [0.75, 0.25, 0.15],  # "self-attention"
    [0.2, 0.85, 0.3],    # "layer"
])

score, sim_matrix = maxsim_score(query_vecs, doc_vecs)

plt.figure(figsize=(6, 4))
sns.heatmap(sim_matrix, annot=True, cmap='viridis', fmt=".2f",
            xticklabels=['self-attention', 'layer'],
            yticklabels=['attention', 'mechanism'])
plt.title('MaxSim Similarity Matrix (Query Tokens vs. Document Tokens)')
plt.xlabel('Document Tokens')
plt.ylabel('Query Tokens')
plt.show()

print(f"\nMaxSim 점수: {score:.4f}")


### 1.4 Single vs Multi-Vector: 복잡도 트레이드오프

> 🔰 **쉽게**: 정밀도를 얻으면 저장소가 1000배 커집니다. 실무에선 압축 기법으로 완화해요.

**점수 공식 한 줄 비교**

| 방식 | 공식 | 의미 |
|---|---|---|
| Single-Vector | `sim(E(q), E(d))` | 문서 전체를 벡터 1개로 압축 |
| **Multi-Vector** | `Σᵢ maxⱼ sim(qᵢ, dⱼ)` | 토큰별 그리디 정렬 |

**복잡도 차이**

| 항목 | Single | Multi |
|---|---|---|
| 저장 | `O(N·d)` | `O(N·T·d)` (T배 ↑) |
| 검색 | `O(\|q\|·N·d)` | `O(\|q\|·N·T·d)` |

**실제 숫자 감 잡기**
- 문서 1,000개 × 평균 1,024 토큰 × 128-dim
- Single: **500KB** / Multi: **500MB** ← 1,000배 차이!

**실무에서 사용 가능한 이유**
1. **PLAID 압축** (ColBERTv2): 1/16까지 축소
2. **Centroid Pruning**: 대표 토큰만 비교
3. **GPU 행렬곱**: MaxSim은 실제로 매우 빠름
4. **품질 게인**: BEIR/DocVQA에서 +10~20점

> 📝 byaldi/colpali-engine은 기본 fp16 저장. 프로덕션에선 PLAID 추천.

### 1.5 Vision Transformer와 Multi-Vector의 만남

> 🔰 **쉽게**: 이미지를 **작은 타일**로 쪼개서 각 타일을 "단어 토큰"처럼 다루면, 텍스트 쿼리와 같은 공간에서 매칭할 수 있어요.

**ViT의 이미지→토큰 변환**
```
입력: 224×224 이미지
      │
      ▼ 16×16 패치로 분할
      ┌─┬─┬─┬─┬─┐
      │ │ │ │ │ │  ← 각 칸이 1개 시각 토큰
      ├─┼─┼─┼─┼─┤  (총 14×14=196개)
      │ │ │ │ │ │
      └─┴─┴─┴─┴─┘
      │
      ▼ 각 패치를 d-dim 벡터로 투영
      (196, d) 행렬
```

결과: 이미지가 **"단어 토큰 시퀀스"처럼** 취급됨. Self-Attention 적용 가능.

**OCR이 불필요한 이유**
- 각 패치 토큰 = 해당 영역의 **시각·공간 정보**(글자·그림·레이아웃) 모두 포함
- 학습 시 "attention diagram" 쿼리 텍스트 ↔ **그림이 그려진 패치 토큰**이 높은 유사도를 갖도록 정렬

> 💡 **비유**: 문서를 모자이크 타일로 쪼개면, 각 타일이 "여기엔 'attention' 단어가 있다"·"여기엔 화살표가 있다"를 아는 벡터로 바뀝니다.

**ColPali 훈련 요약**
| 항목 | 내용 |
|---|---|
| 데이터 | DocVQA, ScreenQA, InfoVQA (문서-질문 쌍) |
| Loss | In-batch Contrastive + MaxSim |
| Backbone | PaliGemma-3B (ColPali) / Qwen2-VL-2B (ColQwen) |
| 튜닝 방식 | LoRA, 생성 head 제거·인코더만 사용 |

**OCR이 못 살리는 정보, ColPali는 보존**
- 📊 표의 병합·셀 경계
- 📈 차트의 축 레이블-데이터 관계
- 📐 수식의 위·아래첨자 구조
- 🇰🇷 한국어·HWP 변환 PDF

### 1.6 ColPali vs ColQwen 한눈에

> 🔰 **쉽게**: 둘 다 ColVision 계열이지만, **한국어·고해상도 문서**라면 ColQwen이 확실히 유리합니다.

| 항목 | ColPali | **ColQwen** |
|---|---|---|
| Vision Encoder | SigLIP (고정 224×224) | **Qwen-Vision (동적 해상도)** |
| Language Model | Gemma-2B (영어 중심) | **Qwen2-2B/7B (다국어)** |
| 한국어 지원 | 제한적 | **우수** |
| 고해상도 처리 | 다운샘플링 필요 | **네이티브 지원** |
| Position Encoding | 1D RoPE | **M-ROPE (3D)** |

→ 상세 이유는 **1.7 절**에서 계속.

### 1.7 Qwen-Vision이 문서를 더 잘 보는 비결: 동적 해상도 + M-ROPE

ColQwen이 한국어 문서에서 특히 강한 이유는 **Qwen-Vision 인코더**가 문서 이미지를 다루는 방식 자체가 다르기 때문입니다. 두 가지 핵심 기술을 살펴봅시다.

---

#### 🔍 ① 동적 해상도 — "원본 그대로 보기"

**기존 ViT의 문제: 사진을 무조건 썸네일로 만든다**

기존 Vision Transformer(ViT)는 입력을 반드시 **224×224 같은 고정 크기**로 맞춰야 합니다. 1920×1080 고해상도 문서 이미지가 들어와도 강제로 축소(다운샘플링)하죠.

> 💡 **비유**: A4 계약서를 명함 크기로 줄여 놓고 "글자 읽어봐"라고 하는 격입니다. 큰 제목은 보여도 본문은 뭉개집니다.

**Qwen-Vision의 해결책: NaViT 방식**

Qwen-Vision은 **원본 해상도를 그대로 받고**, 이미지 크기에 비례해 패치(토큰) 수를 가변적으로 만듭니다.

| 원본 이미지 | 기존 ViT | Qwen-Vision |
|---|---|---|
| 1280×720 (고해상도) | 224×224로 축소 → 196 패치 | **3,600 패치** (원본 유지) |
| 640×480 (중해상도) | 224×224로 축소 → 196 패치 | **1,200 패치** |

> 📊 **결과**: 큰 이미지는 더 많은 토큰으로 세밀하게, 작은 이미지는 적은 토큰으로 효율적으로 처리합니다. 표의 작은 숫자, 차트 범례 같은 **세부 정보가 보존**됩니다.

---

#### 🧭 ② M-ROPE — "이건 어디 위치?"를 알려주는 좌표

Transformer는 원래 "순서"만 알지 "2D 공간 위치"는 모릅니다. 그래서 위치 정보를 인코딩으로 주입해야 하는데, **ColPali는 1D**(선 위 순서)만, **ColQwen은 3D**(시간·세로·가로)를 씁니다.

> 💡 **비유**
> - **ColPali (1D)**: "이 단어는 전체에서 53번째야" 정도만 앎
> - **ColQwen (3D M-ROPE)**: "이 단어는 **페이지 상단**의 **왼쪽 열**에 있는 세 번째 줄이야" 까지 앎

| 축 | 의미 | 문서에서 왜 중요? |
|---|---|---|
| Temporal (시간) | 영상 프레임 순서 | 📄 문서에서는 비활성 |
| **Height (세로)** | 위↔아래 위치 | 📌 헤더·본문·푸터 구분 |
| **Width (가로)** | 왼쪽↔오른쪽 위치 | 📌 2단 레이아웃, 표의 열 구분 |

문서 이미지에서는 Height·Width 두 축만 쓰이는데, 이것만으로도 "**페이지 상단의 제목**", "**왼쪽 열의 표**", "**하단 각주**" 같은 레이아웃 구조를 모델이 인식할 수 있습니다.

---

#### 🇰🇷 ③ 한국어가 특히 잘 되는 이유

두 모델의 **LLM 기반(backbone)** 이 다르다는 점이 결정적입니다.

| 항목 | ColPali | ColQwen |
|---|---|---|
| 기반 LLM | Gemma-2B (**영어 중심**) | Qwen2-2B/7B (**29개 언어, 한국어 포함**) |
| 한국어 토크나이저 | "안녕하세요" → `[안, 녕, 하, 세, 요]` 처럼 단편화 | `[안녕, 하세요]` 처럼 의미 단위로 분할 |
| 사전학습 데이터 | 주로 영문 문서 | 중·영·한 다국어 문서 대량 포함 |

> 💡 **비유**: ColPali는 영어 원어민이 한국어 사전 들고 책 읽는 격이고, ColQwen은 한국어도 모국어처럼 배운 다국어 사용자에 가깝습니다.

**실전 의미**: 한국어 쿼리("삼성전자 3분기 영업이익")와 문서 이미지 속 한국어 텍스트가 **같은 임베딩 공간에 잘 정렬**되므로, 한국 금융·법률·공공문서 검색에서 ColPali보다 확연히 높은 정확도를 보입니다.

---

#### 🎯 한 줄 요약

> **Qwen-Vision = 문서를 원본 그대로 보며(동적 해상도), 2D 좌표를 이해하고(M-ROPE), 한국어를 네이티브로 다루는 인코더**

이 세 가지가 합쳐져서 ColQwen이 한국어 문서 검색의 강력한 선택지가 됩니다.

### 1.8 2026년 ColVision 모델 지형

> 🔰 **쉽게**: 2024년 ColPali 이후 모델이 쏟아졌어요. 목적에 맞는 걸 고르세요.

**주요 공식 모델 (`vidore/*`)**

| 모델 | 기반 | 크기 | 언어 | 특징 |
|---|---|---|---|---|
| `colpali-v1.3-merged` | PaliGemma2 | 3B | 영어 | 영문 최고 성능 |
| `colqwen2-v1.0` | Qwen2-VL | 2B | **한국어 우수** | 29+ 언어 |
| `colqwen2.5-v0.2` | Qwen2.5-VL | 2B | **한국어 우수** | 동적 해상도, 최신 |
| `colSmol-256M/500M` | SmolVLM | ≤500M | 영어 | CPU·엣지용 |

**실험적 (colpali-engine ≥0.3.15)**
- **ColQwen3 / BiQwen3**: Qwen3-VL 기반 (공식 업로드 전)
- **ColQwen-Omni**: 이미지+오디오+비디오 통합 검색

**벤치마크**: **ViDoRe V3** (2026, NVIDIA+ILLUIN) — 26,000+ 페이지, 6개 언어. 사실상 표준.

**상황별 추천**

| 상황 | 선택 |
|---|---|
| 🇰🇷 한국어 문서 | **ColQwen2.5-v0.2** ← 이 노트북 |
| 🇺🇸 영문 성능 최우선 | ColPali-v1.3-merged |
| 💻 GPU 없는 엣지 | ColSmol-500M |
| 🎬 음성·영상 검색 | ColQwen-Omni |

**본 노트북 방침**: Colab T4 검증된 **ColQwen2 v1.0 (byaldi)** + **ColQwen2.5 v0.2 (colpali-engine)**. 모델 ID 한 줄만 바꾸면 다른 모델로 전환 가능.

### 1.9 byaldi vs colpali-engine

> 🔰 **쉽게**: byaldi = **쉬움**. colpali-engine = **정밀**. 이 노트북은 **둘 다** 체험합니다.

| 항목 | **byaldi** | **colpali-engine** |
|---|---|---|
| 개발사 | Answer.AI | Illuin Tech |
| API | RAGatouille 스타일 (쉬움) | HuggingFace 스타일 |
| 인덱스 저장 | 자동 (`.byaldi/`) | 수동 |
| PDF 내장 지원 | ✅ (pdf2image 포함) | ❌ 직접 구현 |
| ColQwen2.5 | ❌ 미지원 | ✅ v0.3.7+ |
| ColQwen3 | ❌ | ✅ (실험) |
| **추천 상황** | **빠른 프로토타입** | **세밀 제어·최신 모델** |

**노트북 구성**
- 섹션 2~3: **byaldi**로 ColQwen2 체험 (쉬운 API)
- 섹션 4: **colpali-engine**으로 ColQwen2.5 전환 (세밀 제어)

---

## 2. byaldi를 활용한 문서 인덱싱

### 2.1 ColQwen 모델 로드

In [ ]:
import torch
from byaldi import RAGMultiModalModel

def get_device():
    """자동 디바이스 감지"""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()
print(f"🔧 사용 디바이스: {device}")

# 로그 레벨 설정
verbose_level = 0 if IN_COLAB else 1

# ColQwen2 v1.0 로드
RAG = RAGMultiModalModel.from_pretrained(
    "vidore/colqwen2-v1.0",
    device=device,
    verbose=verbose_level
)

print(f"✅ ColQwen2 모델 로드 완료 (Device: {device})")

### 2.2 PDF 문서 인덱싱

In [ ]:
from pathlib import Path

# transformer.pdf 경로
pdf_path = Path("transformer.pdf")

# 파일 존재 확인
if not pdf_path.exists():
    print(f"❌ 파일을 찾을 수 없습니다: {pdf_path}")
    print("   data/transformer.pdf 파일을 준비해주세요.")
else:
    print(f"✅ PDF 파일 확인: {pdf_path}")

# 인덱스 생성
index_name = "transformer_colqwen"

try:
    RAG.index(
        input_path=pdf_path,
        index_name=index_name,
        store_collection_with_index=True,  # Base64 이미지 저장 (VLM 통합용)
        overwrite=True,
    )
    print(f"✅ 인덱싱 완료: {index_name}")
except Exception as e:
    print(f"❌ 인덱싱 실패: {e}")

### 2.3 시각적 검색 실행

In [ ]:
# 검색 테스트
query = "attention mechanism diagram"

results = RAG.search(query, k=3)

print(f"🔍 검색 쿼리: {query}")
print("=" * 60)

for i, result in enumerate(results, 1):
    print(f"\n[{i}] 페이지: {result.page_num} | 점수: {result.score:.4f}")

    if hasattr(result, 'base64') and result.base64:
        print(f"    Base64 이미지: {result.base64[:50]}...")

### 2.4 검색 결과 시각화

In [ ]:
import base64
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

def display_search_results(results, max_display=3):
    """검색 결과 이미지 시각화"""
    num_results = min(len(results), max_display)

    if num_results == 0:
        print("검색 결과가 없습니다.")
        return

    fig, axes = plt.subplots(1, num_results, figsize=(6*num_results, 8))

    if num_results == 1:
        axes = [axes]

    for i, result in enumerate(results[:num_results]):
        ax = axes[i]

        if hasattr(result, 'base64') and result.base64:
            image_data = base64.b64decode(result.base64)
            image = Image.open(BytesIO(image_data))

            ax.imshow(image)
            ax.set_title(
                f"Page {result.page_num} (Score: {result.score:.4f})",
                fontsize=12,
                fontweight='bold'
            )
        else:
            ax.text(0.5, 0.5, "이미지 없음", ha='center', va='center')
            ax.set_title(f"Page {result.page_num}")

        ax.axis('off')

    plt.tight_layout()
    plt.show()


# 검색 결과 시각화
display_search_results(results, max_display=3)

### 2.5 다양한 쿼리 테스트

In [ ]:
# 여러 쿼리로 검색 테스트
test_queries = [
    "transformer architecture encoder decoder",
    "multi-head attention",
    "positional encoding",
    "BLEU score results"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 쿼리: {query}")
    print(f"{'='*60}")

    results = RAG.search(query, k=2)

    for i, result in enumerate(results, 1):
        print(f"  [{i}] 페이지 {result.page_num} | 점수: {result.score:.4f}")

    # 검색 결과 시각화
    display_search_results(results, max_display=2)

---

## 3. VLM을 활용한 답변 생성

### 3.1 Retrieve-then-Generate 파이프라인

> 🔰 **쉽게**: ColQwen은 **검색만** 합니다. 답변은 못 해요. 그래서 VLM을 뒤에 붙여야 해요.

**파이프라인 구조**
```
[사용자 질문]
     ↓
📚 ColQwen (Retriever)    ← Top-k 페이지 선택 (MaxSim)
     ↓ (PIL 이미지들)
🤖 Qwen2-VL (Generator)   ← 이미지 보고 답변 생성
     ↓
[최종 답변]
```

> 💡 **비유**: 도서관의 **사서(검색)** vs **교수(답변)** 분업. 사서는 빠르게 책 찾고, 교수는 그 책만 보고 답하면 둘 다 잘합니다.

**왜 분리하나?**

| 이유 | 설명 |
|---|---|
| 🎯 **목적함수 차이** | Retriever=관련도, Generator=유창성·사실성 |
| ♻️ **인덱스 재사용** | 한번 인덱싱한 임베딩은 어떤 VLM과도 결합 가능 |
| 📏 **스케일 차이** | Retriever는 전체 DB, Generator는 Top-k만 |

**지연·비용 구조**

| 단계 | 빈도 | 모델 | 연산 |
|---|---|---|---|
| 인덱싱 | 문서 추가 시 1회 (**오프라인**) | ColQwen 2B | Vision forward |
| 검색 | 쿼리당 | ColQwen 2B (query만) | Text forward + MaxSim |
| 생성 | 쿼리당 | Qwen2-VL 2B (Top-k) | Autoregressive decode |

→ 서비스 지연 = **검색 + 생성** 합계

### 3.2 VLM 아키텍처 기본 구조

> 🔰 **쉽게**: VLM은 이미지와 텍스트를 **같은 언어 공간**으로 통역해서 LLM이 읽게 합니다.

**3단 구성**
```
[이미지]  →  👁 Vision Encoder   →  🔌 Projector  →  ┐
                                                       ├→ 🧠 LLM Decoder → 답변
[텍스트]  →  🔤 Tokenizer   ────────────────────────  ┘
```

| 구성 | 역할 |
|---|---|
| **Vision Encoder** | 이미지 → 패치 토큰 `(n_patches, d_vision)` |
| **Projector** | `d_vision` → `d_lm` 로 차원 맞춤 (2-layer MLP 또는 Q-Former) |
| **LLM Decoder** | 시각+언어 토큰을 하나의 시퀀스로 concat, `<image>` 자리에 패치 투입 |

**프롬프트 예시**
```python
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": pil_image_1},
        {"type": "image", "image": pil_image_2},
        {"type": "text",  "text": "이 두 페이지에서 Attention 구조를 설명해줘"},
    ],
}]
```

**내부 토큰 시퀀스**
```
<|im_start|>user
<|vision_start|> [img1 patches...] <|vision_end|>
<|vision_start|> [img2 patches...] <|vision_end|>
이 두 페이지에서 Attention 구조를 설명해줘
<|im_end|>
<|im_start|>assistant
```

**Qwen2-VL-2B-Instruct 스펙**
- Vision: Qwen-Vision (동적 해상도)
- LLM: Qwen2 1.5B decoder
- Context: 32k tokens
- 4-bit VRAM: ~3GB

### 3.3 Qwen3-VL로의 진화 (2026)

> 🔰 **쉽게**: 2026년 초 공개된 Qwen3-VL은 **긴 영상·작은 글자·수식**에서 크게 개선됐어요.

**라인업**

| 모델 | 아키 | 용도 |
|---|---|---|
| 2B Instruct | Dense | 엣지·Colab T4 |
| 4B Instruct | Dense | 일반 서버 |
| 8B/32B Instruct·Thinking | Dense | 고품질 |
| 30B-A3B | MoE (활성 3B) | 속도·품질 균형 |
| Embedding-2B | 임베딩 전용 | ※ Late Interaction 아님 |

**핵심 개선점 4가지**

| 기술 | 의미 |
|---|---|
| **Interleaved-MRoPE** | 시간축을 교차 배치 → 긴 영상 토큰 의존성 강화 |
| **DeepStack** | Vision 중간 레이어를 LLM에 다단 주입 → 작은 글자·수식 ↑ |
| **Text-Timestamp Alignment** | 영상 토큰-시간 명시 정렬 → "몇 분 장면?" 가능 |
| **Thinking 에디션** | CoT 추론 내장 (o1/Gemini-Thinking과 유사) |

**요구사항**
- `transformers >= 4.57.1` (4.57.0은 yanked)
- 2B Instruct는 T4 + 4-bit 로 실행 가능
- 8B 이상은 A100/H100 권장

**왜 본 노트북은 Qwen2-VL 고정?**
- Colab T4 검증 안정성, transformers 호환성 리스크 최소화
- 개념 학습 단계에선 모델 교체는 부차적 → **섹션 5**에서 Qwen3-VL 교체 실험 제공

### 3.4 Qwen2-VL 모델 로드

In [ ]:
import torch
import gc
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

vlm_model_name = "Qwen/Qwen2-VL-2B-Instruct"

# Colab에서는 4-bit 양자화 사용
use_quantization = IN_COLAB

if use_quantization:
    from transformers import BitsAndBytesConfig

    print("📦 4-bit 양자화 활성화 (Colab 환경)")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,  # T4는 bfloat16 지원 제한적
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
        vlm_model_name,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
else:
    vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
        vlm_model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )

    if device == "mps":
        vlm_model = vlm_model.to(device)

vlm_processor = AutoProcessor.from_pretrained(vlm_model_name)

print(f"✅ Qwen2-VL 모델 로드 완료: {vlm_model_name}")
if torch.cuda.is_available():
    print(f"   현재 GPU 메모리: {torch.cuda.memory_allocated()/1024**3:.2f}GB 사용 중")

### 3.5 NF4와 Double Quantization

> 🔰 **쉽게**: 4-bit로 압축하면 모델이 1/3로 작아져요. 비결은 **가중치 분포에 맞는 양자화**(NF4) + **스케일 상수도 압축**(Double Quant).

**참고**: QLoRA 논문 (Dettmers et al., **NeurIPS 2023**)

---

**① NF4 (NormalFloat 4-bit)**

- 일반 Int4: `[-8, -7, ..., 7]` **균등** 16단계
- **NF4**: 가중치가 **정규분포 N(0,σ²)** 를 따른다는 관찰 → 16단계를 **정규분포 quantile**로 배치
- 결과: Int4 대비 정확도 손실 **절반**

> 💡 **비유**: 옷 사이즈를 1~16 균등(S/M/L만 16칸)으로 자르면 어색. **실제 키 분포**에 맞춰 16단계 정하면 훨씬 잘 맞아요. NF4가 후자.

---

**② Double Quantization**

- 4-bit 저장 시 블록(64~256개)마다 **스케일 상수** (fp16/32) 필요 → 이 상수 자체를 또 8-bit로 양자화
- 추가 절약: **~0.4 bits/param** (Qwen2-VL-2B 기준 ~100MB)

---

**③ compute_dtype 선택**
```python
bnb_4bit_compute_dtype=torch.float16,   # T4 환경
```

| GPU | 권장 dtype |
|---|---|
| Turing (T4) | `float16` |
| Ampere+ (A100, H100) | `bfloat16` (수치 안정성 ↑) |

---

**VRAM 절약 효과 (Qwen2-VL-2B)**

| 설정 | VRAM |
|---|---|
| FP16 | ~4.5 GB |
| 4-bit (NF4만) | ~1.8 GB |
| **4-bit + Double Quant** | **~1.5 GB** |

→ T4(16GB)에 ColQwen + Qwen2-VL **동시 로드** 가능 (~7GB 사용, 여유 충분)

**실무 팁**
- Qwen-VL 계열은 4-bit에서도 성능 저하 미미 → 4-bit가 표준
- QLoRA 파인튜닝 기본 조합: `nf4 + double_quant`

### 3.6 RAG 파이프라인 구현

In [ ]:
import io
import base64
import gc
from PIL import Image
from qwen_vl_utils import process_vision_info

def multimodal_rag_pipeline(query: str, top_k: int = 2, max_new_tokens: int = 256):
    """
    멀티모달 RAG 파이프라인: ColQwen 검색 + Qwen2-VL 생성

    Args:
        query: 검색 쿼리
        top_k: 검색할 상위 문서 수 (Colab에서는 1-2 권장)
        max_new_tokens: 생성할 최대 토큰 수 (메모리 절약을 위해 256 권장)

    Returns:
        생성된 답변 문자열
    """
    try:
        # 1. ColQwen 검색
        print(f"🔍 검색 중: {query}")
        search_results = RAG.search(query, k=top_k)

        if not search_results:
            return "❌ 검색 결과가 없습니다. 다른 키워드로 시도해보세요."

        print(f"✅ {len(search_results)}개 페이지 검색 완료")
        for i, result in enumerate(search_results):
            print(f"   [{i+1}] 페이지 {result.page_num}, 점수: {result.score:.4f}")

        # 2. 검색된 이미지 준비 (해상도 제한으로 메모리 절약)
        images = []
        max_size = (640, 640) if IN_COLAB else (1024, 1024)  # Colab에서 해상도 더 제한

        for result in search_results:
            if hasattr(result, 'base64') and result.base64:
                image_data = base64.b64decode(result.base64)
                image = Image.open(io.BytesIO(image_data)).convert("RGB")

                # 이미지 리사이즈 (메모리 절약)
                image.thumbnail(max_size, Image.Resampling.LANCZOS)
                images.append(image)

        if not images:
            return "❌ 검색 결과에 이미지가 없습니다. store_collection_with_index=True로 인덱싱하세요."

        # 3. VLM 입력 구성
        messages = [
            {
                "role": "user",
                "content": [
                    *[{"type": "image", "image": img} for img in images],
                    {"type": "text", "text": f"Based on the document images, answer the question.\n\nQuestion: {query}\n\nAnswer:"}
                ]
            }
        ]

        # 4. 답변 생성
        print("🤖 답변 생성 중...")

        # 메모리 정리
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vlm_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(vlm_model.device)

        # 중간 변수 정리
        del image_inputs, video_inputs, images, messages
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        with torch.no_grad():
            generated_ids = vlm_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # greedy decoding (메모리 절약)
                pad_token_id=vlm_processor.tokenizer.pad_token_id,
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        response = vlm_processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        # GPU 메모리 정리
        del inputs, generated_ids, generated_ids_trimmed
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("✅ 답변 생성 완료")
        return response

    except torch.cuda.OutOfMemoryError:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return "❌ GPU 메모리 부족. top_k=1로 줄이거나 런타임을 재시작하세요."
    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"❌ 오류 발생: {type(e).__name__}: {str(e)}"

### 3.7 RAG 파이프라인 실행

In [ ]:
# 질문 예시 (Colab에서는 top_k=1로 메모리 절약)
query = "What is the attention mechanism in transformers?"

# Colab에서는 이미지 1개만 사용하여 메모리 절약
top_k_value = 1 if IN_COLAB else 3
answer = multimodal_rag_pipeline(query, top_k=top_k_value, max_new_tokens=200)

print("\n" + "="*60)
print(f"❓ 질문: {query}")
print("="*60)
print(f"\n💡 답변:\n{answer}")

In [ ]:
# 여러 질문 테스트 (Colab에서는 top_k=1로 더 줄임)
test_questions = [
    "How does multi-head attention work?",
    "What is positional encoding?",
]

top_k_value = 1 if IN_COLAB else 2

for question in test_questions:
    print("\n" + "="*80)
    print(f"❓ 질문: {question}")
    print("="*80)

    answer = multimodal_rag_pipeline(question, top_k=top_k_value)
    print(f"\n💡 답변:\n{answer}")

---

## 4. colpali-engine으로 ColQwen2.5 전환

**colpali-engine**은 최신 ColQwen2.5 모델을 지원합니다.

In [ ]:
import torch
import gc
from PIL import Image
from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor

# Colab에서는 이전 모델 정리 필요
if IN_COLAB:
    print("🧹 이전 모델 메모리 정리 중...")

    # VLM 모델 정리
    if 'vlm_model' in dir():
        del vlm_model
        print("   - Qwen2-VL 모델 삭제")

    if 'vlm_processor' in dir():
        del vlm_processor

    # byaldi RAG 모델 정리
    if 'RAG' in dir():
        del RAG
        print("   - ColQwen2 (byaldi) 모델 삭제")

    gc.collect()
    torch.cuda.empty_cache()
    print(f"   GPU 메모리 정리 완료: {torch.cuda.memory_allocated()/1024**3:.2f}GB 사용 중")

# ColQwen2.5 모델 로드
colqwen25_device = device
print(f"\n🔧 ColQwen2.5 디바이스: {colqwen25_device}")

# Colab T4는 bfloat16 지원이 제한적이므로 float16 사용
if IN_COLAB:
    model_colqwen25 = ColQwen2_5.from_pretrained(
        "vidore/colqwen2.5-v0.2",
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    ).eval()
else:
    model_colqwen25 = ColQwen2_5.from_pretrained(
        "vidore/colqwen2.5-v0.2",
        torch_dtype=torch.bfloat16 if colqwen25_device != "cpu" else torch.float32,
        device_map=colqwen25_device if colqwen25_device == "cuda" else None,
    ).eval()

    if colqwen25_device != "cuda":
        model_colqwen25 = model_colqwen25.to(colqwen25_device)

processor_colqwen25 = ColQwen2_5_Processor.from_pretrained("vidore/colqwen2.5-v0.2")

print("✅ ColQwen2.5 모델 로드 완료")
if torch.cuda.is_available():
    print(f"   현재 GPU 메모리: {torch.cuda.memory_allocated()/1024**3:.2f}GB 사용 중")

In [ ]:
from pdf2image import convert_from_path

# PDF를 이미지로 변환 (Colab에서는 낮은 DPI로 메모리 절약)
dpi_value = 100 if IN_COLAB else 150
pdf_images = convert_from_path(str(pdf_path), dpi=dpi_value)
print(f"✅ {len(pdf_images)}개 페이지 변환 완료 (DPI: {dpi_value})")

# 문서 임베딩 생성
print("📄 문서 임베딩 생성 중...")
doc_embeddings = []

with torch.no_grad():
    for i, img in enumerate(pdf_images):
        batch_images = processor_colqwen25.process_images([img]).to(model_colqwen25.device)
        embedding = model_colqwen25(**batch_images)

        # CPU로 이동하여 GPU 메모리 절약
        doc_embeddings.append(embedding.cpu())

        # 중간 변수 정리
        del batch_images, embedding

        if (i + 1) % 5 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"   {i + 1}/{len(pdf_images)} 페이지 완료")

print(f"✅ 총 {len(doc_embeddings)}개 페이지 임베딩 완료")
if torch.cuda.is_available():
    print(f"   현재 GPU 메모리: {torch.cuda.memory_allocated()/1024**3:.2f}GB 사용 중")

In [ ]:
def search_colqwen25(query: str, top_k: int = 3):
    """
    ColQwen2.5를 사용한 시각적 검색

    Args:
        query: 검색 쿼리
        top_k: 반환할 상위 결과 수

    Returns:
        검색 결과 리스트 [{"page": int, "score": float, "image": PIL.Image}, ...]
    """
    with torch.no_grad():
        batch_queries = processor_colqwen25.process_queries([query]).to(model_colqwen25.device)
        query_embedding = model_colqwen25(**batch_queries)

    # MaxSim 점수 계산 (임베딩을 같은 디바이스로 이동)
    scores = []
    for doc_emb in doc_embeddings:
        # doc_emb가 CPU에 있으면 GPU로 이동
        doc_emb_device = doc_emb.to(model_colqwen25.device)
        score = processor_colqwen25.score_multi_vector(query_embedding, doc_emb_device)[0].item()
        scores.append(score)
        del doc_emb_device  # 메모리 정리

    # 상위 k개 결과
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    # 메모리 정리
    del batch_queries, query_embedding
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return [
        {"page": idx + 1, "score": scores[idx], "image": pdf_images[idx]}
        for idx in top_indices
    ]


# 검색 테스트
query = "attention mechanism"
results = search_colqwen25(query, top_k=3)

print(f"🔍 쿼리: {query}")
print("=" * 60)
for r in results:
    print(f"   페이지 {r['page']}: {r['score']:.4f}")

In [ ]:
# 검색 결과 시각화
import matplotlib.pyplot as plt

num_results = min(len(results), 3)  # 최대 3개만 표시
fig, axes = plt.subplots(1, num_results, figsize=(5*num_results, 7))

if num_results == 1:
    axes = [axes]

for i, r in enumerate(results[:num_results]):
    axes[i].imshow(r['image'])
    axes[i].set_title(f"Page {r['page']} (Score: {r['score']:.4f})")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# 메모리 정리
plt.close(fig)
gc.collect()

---

## 연습문제

1. **다른 쿼리 테스트**
   - "encoder decoder architecture" 쿼리로 검색해보세요.
   - 예상되는 페이지가 검색되나요?

2. **검색 정확도 분석**
   - ColQwen2 (byaldi)와 ColQwen2.5 (colpali-engine)의 검색 결과를 비교해보세요.

3. **한국어 문서 테스트**
   - 한국어 PDF 문서로 동일한 파이프라인을 구축해보세요.
   - 한국어 쿼리의 검색 성능은 어떤가요?

4. **성능 최적화**
   - 문서 임베딩을 파일로 저장하고 로드하는 코드를 구현하세요.
   - 대용량 문서 처리 시 어떤 최적화가 필요할까요?

In [ ]:

# 연습문제 코드를 여기에 작성하세요

